# Same trick on DeepSeek — `deepseek-flash` (DeepSeek-V4.1-Flash) over the hosted API

Twin of `real-llm.ipynb`: same input dict, same prompt text, same renormalise-over-candidates. What changes is the platform. Probed live on 2026-09-24:

| llama.cpp (`minijev`) | DeepSeek API | consequence |
|---|---|---|
| raw prompt, `Answer:` prefix | chat `messages` only | put the question in the user turn |
| — | thinking **on** by default | `thinking: {"type": "disabled"}` or the first token is reasoning (`"We"`) |
| `/tokenize` → ids | no tokenize endpoint | match candidates by **token text**, not id |
| `logit_bias` +50 | **silently ignored** (proved below) | no forcing — the pick is still in-schema because we argmax over *our* candidates, but a candidate the model hates can fall out of the report |
| `n_probs` up to 40, `post_sampling_probs` | `top_logprobs` **max 20**, reports post-temperature | a candidate outside the top 20 → raise, never score it 0 |
| `temperature=1.0` | same — at 0 every non-top token reads `-9999` | keep 1.0 |

The math is unchanged: equal bias cancelled anyway, so renormalising the raw top-20 over the candidates gives the same relative belief — as long as every candidate is in the report.

In [1]:
import json, math, os, sys, time, urllib.request, urllib.error
HERE = os.getcwd()
ROOT = os.path.dirname(HERE) if os.path.basename(HERE) == "learn" else HERE
sys.path.insert(0, ROOT)
import minijev  # loads ROOT/.env (DEEPSEEK_API_KEY lives there); reused for _marks + confidence

DS_URL = "https://api.deepseek.com"
MODEL = "deepseek-flash"
DS_KEY = os.environ["DEEPSEEK_API_KEY"]

def ds_post(path, body):
    req = urllib.request.Request(DS_URL + path, json.dumps(body).encode(),
                                 {"Content-Type": "application/json", "Authorization": "Bearer " + DS_KEY})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            return json.load(r)
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode()[:500]}") from None

t0 = time.perf_counter()
req = urllib.request.Request(DS_URL + "/models", headers={"Authorization": "Bearer " + DS_KEY})
models = {m["id"]: m.get("name") for m in json.load(urllib.request.urlopen(req, timeout=30))["data"]}
print(f"API OK in {(time.perf_counter() - t0) * 1000:.0f} ms; models: {models}")
assert MODEL in models

API OK in 284 ms; models: {'deepseek-flash': 'DeepSeek-V4.1-Flash', 'deepseek-v4-pro': 'DeepSeek-V4-Pro'}


## Step 0 — input dict

Identical to `real-llm.ipynb`.

In [2]:
INPUT = {
    "state": "Sorry about the outage -- we have reset everyone's limits for the day.",
    "questions": {
        "is_quota_reset": {"type": "noul", "instructions": "Does this announce a quota reset?"},
        "urgency": {"type": "choice", "instructions": "How urgent is this?",
                    "criteria": {"ignore": "not relevant", "today": "act today",
                                 "now": "stop what you are doing"}},
    },
}
print(json.dumps(INPUT, indent=2))

{
  "state": "Sorry about the outage -- we have reset everyone's limits for the day.",
  "questions": {
    "is_quota_reset": {
      "type": "noul",
      "instructions": "Does this announce a quota reset?"
    },
    "urgency": {
      "type": "choice",
      "instructions": "How urgent is this?",
      "criteria": {
        "ignore": "not relevant",
        "today": "act today",
        "now": "stop what you are doing"
      }
    }
  }
}


## Step 1 — dict → chat messages

Same body text as `minijev.choice`. No `<|im_start|>` template and no `Answer:` prefix: the hosted API owns the chat template, so the question goes in a plain user message. Labels are the **bare** marks (`"2"`, not `" 2"`) — with thinking off, the answer is the first token of the reply, no leading space.

In [3]:
state = INPUT["state"]
q = INPUT["questions"]["urgency"]
keys = list(q["criteria"])
marks = minijev._marks(len(keys))
menu = "\n".join(m + ". " + k + " -- " + q["criteria"][k] for m, k in zip(marks, keys))
body = ("Text:\n" + state + "\n\nQuestion: " + q["instructions"] + "\nOptions:\n" + menu
        + f"\nReply with exactly one character: {', '.join(marks)}.")
messages = [{"role": "user", "content": body}]
labels = {k: m for k, m in zip(keys, marks)}
print(body)
print("-"*20)
print("labels:", labels)

Text:
Sorry about the outage -- we have reset everyone's limits for the day.

Question: How urgent is this?
Options:
1. ignore -- not relevant
2. today -- act today
3. now -- stop what you are doing
Reply with exactly one character: 1, 2, 3.
--------------------
labels: {'ignore': '1', 'today': '2', 'now': '3'}


## Step 2 — request body + raw response

`max_tokens=1`, thinking off, `logprobs` + `top_logprobs=20` (the API's ceiling), `temperature=1.0`. No `logit_bias` — see the proof cell further down. Every row of the raw report is printed so you can see where the candidates sit.

In [4]:
req_body = {
    "model": MODEL,
    "messages": messages,
    "max_tokens": 1,
    "temperature": 1.0,
    "thinking": {"type": "disabled"},
    "logprobs": True,
    "top_logprobs": 20,
}
print(json.dumps({k: v for k, v in req_body.items() if k != "messages"}, indent=2))
print("-"*20)

t0 = time.perf_counter()
out = ds_post("/chat/completions", req_body)
print(f"served in {(time.perf_counter() - t0) * 1000:.0f} ms; usage: {out['usage']}")
choice0 = out["choices"][0]
print("sampled token:", repr(choice0["message"]["content"]), "| finish:", choice0["finish_reason"])
report = choice0["logprobs"]["content"][0]["top_logprobs"]
for r in report:
    print(f"   {r['token']!r:12} logprob={r['logprob']:10.4f}  p={math.exp(r['logprob']):.6f}")

{
  "model": "deepseek-flash",
  "max_tokens": 1,
  "temperature": 1.0,
  "thinking": {
    "type": "disabled"
  },
  "logprobs": true,
  "top_logprobs": 20
}
--------------------


served in 416 ms; usage: {'prompt_tokens': 69, 'completion_tokens': 1, 'total_tokens': 70, 'prompt_tokens_details': {'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 69}
sampled token: '2' | finish: length
   '2'          logprob=   -0.0846  p=0.918898
   '1'          logprob=   -2.7410  p=0.064505
   '3'          logprob=   -4.1545  p=0.015693
   'The'        logprob=   -7.3548  p=0.000639
   'I'          logprob=  -10.2355  p=0.000036
   'This'       logprob=  -10.4623  p=0.000029
   'Based'      logprob=  -10.5493  p=0.000026
   'No'         logprob=  -10.7255  p=0.000022
   'It'         logprob=  -10.8781  p=0.000019
   'Since'      logprob=  -11.1663  p=0.000014
   'To'         logprob=  -11.7057  p=0.000008
   'What'       logprob=  -11.8876  p=0.000007
   'If'         logprob=  -11.9233  p=0.000007
   'There'      logprob=  -12.1785  p=0.000005
   'A'          logprob=  -12.2137  p=0.000005
   'Let'        logprob=  -12.3240  p=0.000004
   'Sorry'  

## Step 3 — match by token text, renormalise

No ids, so a candidate is every report row whose text, stripped and case-folded, equals the label — `"Yes"`, `" Yes"`, `"yes"`, `"YES"` are all the same answer, so their probabilities are **summed**. Then softmax over the candidates only. A candidate with no row at all is outside the top 20: raise instead of calling it zero.

In [5]:
def candidate_probs(report, labels):
    norm = lambda s: s.strip().casefold()
    raw = {name: 0.0 for name in labels}
    for r in report:
        for name, tok in labels.items():
            if norm(r["token"]) == norm(tok):
                raw[name] += math.exp(r["logprob"])
    missing = [n for n, v in raw.items() if v == 0.0]
    if missing:   # outside the top-20 window -- refuse rather than invent a number
        raise RuntimeError(f"candidates missing from the top-20 report: {missing}")
    z = sum(raw.values())
    return {k: v / z for k, v in raw.items()}, z

probs, mass = candidate_probs(report, labels)
print("renormalised:", {k: round(v, 4) for k, v in probs.items()})
print(f"candidate mass before renormalising: {mass:.4f}  (llama.cpp + bias forces ~1.0; here it is the model's own)")
print("pick:", max(probs, key=probs.get), "| confidence:", minijev.confidence(probs))

renormalised: {'ignore': 0.0646, 'today': 0.9197, 'now': 0.0157}
candidate mass before renormalising: 0.9991  (llama.cpp + bias forces ~1.0; here it is the model's own)
pick: today | confidence: 0.8552


## Proof — `logit_bias` is accepted and ignored

**How to read this cell: the digits *staying* is the proof.** We send a bias that, if the API applied it, would make `"1"` win outright (+100 on `"1"`, −100 on `"2"` and `"3"`). On llama.cpp that is exactly what `minijev` relies on.

Token ids come from DeepSeek's published tokenizer (`deepseek-ai/DeepSeek-V4-Flash/tokenizer.json` on HuggingFace, same as V3): `"1"`=19, `"2"`=20, `"3"`=21.

| if the bias were applied | what we get |
|---|---|
| reported probs: `"1"` ≈ 1.0 | unchanged — `"2"` still on top |
| sampled token: `"1"` every time | same mix as with no bias |

The request still returns 200: no error, no warning. So the llama.cpp guarantee — *a value outside your schema is unrepresentable* — does **not** exist on this API. We keep the schema only by reading our own candidates out of the report.

In [6]:
import collections
DIGIT_ID = {"1": 19, "2": 20, "3": 21}   # from DeepSeek-V4-Flash tokenizer.json
bias = {str(DIGIT_ID["1"]): 100, str(DIGIT_ID["2"]): -100, str(DIGIT_ID["3"]): -100}
print("logit_bias sent:", bias, "  -> if honoured, the answer must be '1'")

biased = ds_post("/chat/completions", {**req_body, "logit_bias": bias})
b_report = biased["choices"][0]["logprobs"]["content"][0]["top_logprobs"]
fmt = lambda rep: [(r["token"], round(math.exp(r["logprob"]), 4)) for r in rep[:4]]
print("\nreported probs, with bias   :", fmt(b_report))
print("reported probs, without bias:", fmt(report))

# the report could in principle be pre-bias (llama.cpp's default is), so also check what gets SAMPLED
N = 10
sampled = collections.Counter(ds_post("/chat/completions", {**req_body, "logit_bias": bias})["choices"][0]["message"]["content"]
                              for _ in range(N))
print(f"\nsampled token over {N} biased calls: {dict(sampled)}   (honoured bias would give {{'1': {N}}})")
print("verdict:", "bias APPLIED" if sampled.get("1", 0) == N else "bias IGNORED")

logit_bias sent: {'19': 100, '20': -100, '21': -100}   -> if honoured, the answer must be '1'



reported probs, with bias   : [('2', 0.9578), ('3', 0.0367), ('1', 0.0049), ('The', 0.0004)]
reported probs, without bias: [('2', 0.9189), ('1', 0.0645), ('3', 0.0157), ('The', 0.0006)]



sampled token over 10 biased calls: {'2': 10}   (honoured bias would give {'1': 10})
verdict: bias IGNORED


## Step 4 — the one-liners, and a side-by-side with the local bonsai

`decide()` wraps steps 2–3. `noul` / `choice` / `score` reuse `minijev`'s exact body text, so the only variable between the two columns is the model. Server nondeterminism note: DeepSeek's logprobs are **not** repeatable — the identical urgency prompt read `today` 0.93 in step 3 and 0.97 here (batching on shared GPUs). Average a few calls if you need a stable number; llama.cpp on one box is bit-stable.

**Reading `noul (1.0, 1.0)`:** that tuple is `(P(true), confidence)` rounded to 4 dp. Unrounded, DeepSeek puts `No` at logprob ≈ −19.7, i.e. P(false) ≈ 1e-9 — real, not a bug. DeepSeek's first-token distribution is simply much **sharper** than bonsai's: on deliberately vague texts it also answers ~0.000 / ~1.000, where bonsai gives 0.06 / 0.01. So the probabilities work mechanically, but as *graded* uncertainty they are near one-hot — treat DeepSeek's confidence as a vote, not a calibrated measure, unless you calibrate it against labelled data.

In [7]:
def decide(body, labels):
    out = ds_post("/chat/completions", {
        "model": MODEL, "messages": [{"role": "user", "content": body}],
        "max_tokens": 1, "temperature": 1.0, "thinking": {"type": "disabled"},
        "logprobs": True, "top_logprobs": 20,
    })
    return candidate_probs(out["choices"][0]["logprobs"]["content"][0]["top_logprobs"], labels)[0]

def noul(state, question, criteria=None):
    crit = f"\ntrue means: {criteria['true']}\nfalse means: {criteria['false']}" if criteria else ""
    body = f"Text:\n{state}\n\nQuestion: {question}{crit}\nReply with exactly one word: Yes or No."
    p = decide(body, {"true": "Yes", "false": "No"})
    return p["true"], minijev.confidence(p)

def choice(state, question, options):
    keys = list(options)
    marks = minijev._marks(len(keys))
    menu = "\n".join(f"{m}. {k} -- {options[k]}" for m, k in zip(marks, keys))
    body = (f"Text:\n{state}\n\nQuestion: {question}\nOptions:\n{menu}\n"
            f"Reply with exactly one character: {', '.join(marks)}.")
    p = decide(body, dict(zip(keys, marks)))
    return p, minijev.confidence(p)

def score(state, question, levels):
    marks = minijev._marks(len(levels))
    menu = "\n".join(f"{m}. {d}" for m, d in zip(marks, levels))
    body = (f"Text:\n{state}\n\nQuestion: {question}\nScale:\n{menu}\n"
            f"Reply with exactly one character: {', '.join(marks)}.")
    p = decide(body, {str(i + 1): m for i, m in enumerate(marks)})
    return sum(int(k) * v for k, v in p.items()), p, minijev.confidence(p)

LEVELS = ["no impact", "minor inconvenience", "noticeable disruption", "serious outage", "total outage"]
SCORE_Q = "How severe was the incident being apologised for?"

t0 = time.perf_counter()
ds = {"noul": noul(state, INPUT["questions"]["is_quota_reset"]["instructions"]),
      "choice": choice(state, q["instructions"], q["criteria"]),
      "score": score(state, SCORE_Q, LEVELS)}
print(f"deepseek-flash: 3 calls in {(time.perf_counter() - t0) * 1000:.0f} ms")

try:
    t0 = time.perf_counter()
    local = {"noul": minijev.noul(state, INPUT["questions"]["is_quota_reset"]["instructions"]),
             "choice": minijev.choice(state, q["instructions"], q["criteria"]),
             "score": minijev.score(state, SCORE_Q, LEVELS)}
    print(f"bonsai (local): 3 calls in {(time.perf_counter() - t0) * 1000:.0f} ms")
except Exception as e:
    local = None
    print("local bonsai unavailable:", e)

r4 = lambda d: {k: round(v, 4) for k, v in d.items()}
# P(false) printed separately: rounded to 4 dp, P(true)=0.999999999 shows as 1.0 and looks like a bug
rows = [("noul P(true) / P(false)", lambda x: (round(x["noul"][0], 4), f"P(false)={1 - x['noul'][0]:.1e}", x["noul"][1])),
        ("choice",       lambda x: (max(x["choice"][0], key=x["choice"][0].get), r4(x["choice"][0]), x["choice"][1])),
        ("score E[level]", lambda x: (round(x["score"][0], 3), r4(x["score"][1]), x["score"][2]))]
for name, f in rows:
    print(f"\n{name}")
    print("   deepseek-flash:", f(ds))
    if local:
        print("   bonsai-2-27b  :", f(local))

deepseek-flash: 3 calls in 1589 ms


bonsai (local): 3 calls in 1129 ms

noul P(true) / P(false)
   deepseek-flash: (1.0, 'P(false)=2.3e-09', 1.0)
   bonsai-2-27b  : (0.9839, 'P(false)=1.6e-02', 0.9678)

choice
   deepseek-flash: ('today', {'ignore': 0.0089, 'today': 0.9335, 'now': 0.0576}, 0.8759)
   bonsai-2-27b  : ('today', {'ignore': 0.2573, 'today': 0.62, 'now': 0.1227}, 0.3627)

score E[level]
   deepseek-flash: (3.994, {'1': 0.0, '2': 0.0001, '3': 0.0068, '4': 0.9922, '5': 0.0009}, 0.9853)
   bonsai-2-27b  : (3.638, {'1': 0.0233, '2': 0.0844, '3': 0.1556, '4': 0.7044, '5': 0.0323}, 0.5488)
